## Import libraries and extract data

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn import preprocessing

raw_data = np.loadtxt('Audiobooks_data.csv', delimiter=',')

unscaled_inputs = raw_data[:, 1:-1]
targets = raw_data[:, -1]

#### Balance the dataset

In [ ]:
one_targets = int(np.sum(targets))
zero_targets = 0
indices_to_remove = []

for i in range(targets.shape[0]):
    if targets[i] == 0:
        if zero_targets < one_targets:
            zero_targets += 1
        else:
            indices_to_remove.append(i)
print('Number of indices to remove: ', len(indices_to_remove))

unscaled_inputs_balanced = np.delete(unscaled_inputs, indices_to_remove, axis=0)
targets_balanced = np.delete(targets, indices_to_remove, axis=0)

print('Targets after balancing: {0:2f} // Inputs after balancing: {1:2f} '.format(targets_balanced.shape[0], unscaled_inputs_balanced.shape[0]))

### Class distribution: before vs. after balancing

In [ ]:
labels = ['Will not return (0)', 'Will return (1)']
before = [int(np.sum(targets == 0)), int(np.sum(targets == 1))]
after  = [int(np.sum(targets_balanced == 0)), int(np.sum(targets_balanced == 1))]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, before, width, label='Before balancing', color='#e07b7b')
bars2 = ax.bar(x + width/2, after,  width, label='After balancing',  color='#6baed6')

ax.set_title('Class Distribution Before and After Balancing', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of samples')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

### Standardize the inputs

In [ ]:
scaled_inputs = preprocessing.scale(unscaled_inputs_balanced)

### Feature correlation heatmap

In [ ]:
feature_names = [
    'Book Length (overall)',
    'Book Length (avg)',
    'Price (overall)',
    'Price (avg)',
    'Review',
    'Review Score',
    'Total Minutes Listened',
    'Completion',
    'Support Requests',
    'Last Interaction'
]

df = pd.DataFrame(unscaled_inputs_balanced, columns=feature_names)
df['Target'] = targets_balanced

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df.corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Shuffle the data

In [ ]:
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_balanced[shuffled_indices]

### Split the dataset into train, validation and test sets

In [ ]:
sample_count = shuffled_inputs.shape[0]

train_samples_count = int(0.8 * sample_count)
validation_samples_count = int(0.1 * sample_count)
test_samples_count = sample_count - train_samples_count - validation_samples_count

train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

validation_inputs = shuffled_inputs[train_samples_count: train_samples_count + validation_samples_count]
validation_targets = shuffled_targets[train_samples_count: train_samples_count + validation_samples_count]

test_inputs = shuffled_inputs[train_samples_count + validation_samples_count:]
test_targets = shuffled_targets[train_samples_count + validation_samples_count:]

print('Training set: {0} samples // Validation set: {1} samples // Test set: {2} samples'.format(train_inputs.shape[0], validation_inputs.shape[0], test_inputs.shape[0]))


### Save the datasets into .npz files

In [ ]:
np.savez('Audiobooks_train.npz', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_validation.npz', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_test.npz', inputs=test_inputs, targets=test_targets)